## Task 1 — Import and prepare the data

The three files are large (19.7, 35.6 and 5.1 million rows), so each file is read inside a function that
keeps only the columns and rows needed, and returns only the filtered result. The full file therefore only exists
in memory while the function runs.

* `202608_EPO_IPC.txt` has one row **per IPC code**, so the same patent appears on several rows. We keep
  `appln_id` and `prio_year`, restrict to priority years 2000–2019 and drop duplicate rows → one row per patent.
* `202608_CPC_Classes.txt` is the largest file. It is read in chunks, and from each chunk we keep only Y02E codes
  of patents in the sample. All other CPC codes are never needed.
* `202608_EPO_App_reg.txt` is also read in chunks and restricted to the sample. Applicants linked to several
  regions appear on several rows that are identical apart from the region columns; once only the needed columns
  are read, these rows are exact duplicates and are dropped.

The CPC codes in the file are written with two spaces after `Y02E`, e.g. `Y02E  10/549`, so the main group is
characters 6–7 and the first digit after the slash is character 9.

In [1]:
import os

import numpy as np
import pandas as pd

In [2]:
# Constants used throughout the notebook
DATA_DIR = "data/"
# Stop with a clear message if the data folder is not next to the notebook
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"No folder {DATA_DIR!r} in {os.getcwd()}. Put the data folder next to the notebook.")
FIRST_YEAR, LAST_YEAR = 2000, 2019   # sample period (priority years)
CHUNK_SIZE = 2_000_000               # rows per chunk when reading the large files

# Table 1: main groups of Y02E
MAIN_GROUPS = {
    "10": "Renewable energy",
    "20": "Efficient combustion",
    "30": "Nuclear energy",
    "40": "Efficient power transmission",
    "50": "Non-fossil fuels",
    "60": "Storage and hydrogen",
    "70": "Other energy conversion",
}

# Table 2: technologies within main group 10, keyed by the first digit after the slash
TECHNOLOGIES = {
    "4": "Solar", "5": "Solar", "6": "Solar",
    "7": "Wind",
    "1": "Other renewables", "2": "Other renewables", "3": "Other renewables",
}

In [3]:
def read_sample(path):
    """Return one row per EPO patent application with priority year in the sample period."""
    df = pd.read_csv(path, sep="|", usecols=["appln_id", "prio_year"])
    # The file has one row per IPC code: keep the sample years, then drop the repeated rows
    df = df[df["prio_year"].between(FIRST_YEAR, LAST_YEAR)].drop_duplicates()
    # Smaller integer types: the ids fit in int32 and the years in int16
    return df.astype({"appln_id": "int32", "prio_year": "int16"}).reset_index(drop=True)

In [4]:
def read_green_codes(path, sample_ids):
    """Return the Y02E codes of patents in the sample, reading the file in chunks."""
    parts = []
    for chunk in pd.read_csv(path, sep="|", usecols=["appln_id", "CPC_Class"],
                             dtype={"appln_id": "int32", "CPC_Class": "str"},
                             chunksize=CHUNK_SIZE):
        # Keep only green energy codes (Y02E) of patents in the sample
        keep = chunk["CPC_Class"].str.startswith("Y02E") & chunk["appln_id"].isin(sample_ids)
        parts.append(chunk[keep])
    return pd.concat(parts, ignore_index=True)

In [5]:
def read_applicants(path, sample_ids):
    """Return the applicants of patents in the sample, with region duplicates removed."""
    parts = []
    # utf-8-sig removes the byte order mark at the start of this file.
    # keep_default_na=False + na_values=[""]: only empty fields are missing, so that the
    # country code "NA" (Namibia) is not read as a missing value
    for chunk in pd.read_csv(path, sep="|", encoding="utf-8-sig",
                             keep_default_na=False, na_values=[""],
                             usecols=["appln_id", "app_name", "ctry_code", "app_share"],
                             dtype={"appln_id": "int32", "app_name": "str", "ctry_code": "str"},
                             chunksize=CHUNK_SIZE):
        parts.append(chunk[chunk["appln_id"].isin(sample_ids)])
    df = pd.concat(parts, ignore_index=True)
    # Rows that differed only in the (unread) region columns are now exact duplicates
    return df.drop_duplicates().reset_index(drop=True)

In [6]:
# Read the three files
patents = read_sample(DATA_DIR + "202608_EPO_IPC.txt")
codes = read_green_codes(DATA_DIR + "202608_CPC_Classes.txt", patents["appln_id"])
applicants = read_applicants(DATA_DIR + "202608_EPO_App_reg.txt", patents["appln_id"])

In [7]:
# Add main group and technology to each Y02E code (parsed once here, reused in Tasks 3–5)
codes["main_group"] = codes["CPC_Class"].str[6:8].map(MAIN_GROUPS)
is_renewable = codes["main_group"] == "Renewable energy"
# Technology only applies to main group 10; codes not in Table 2 get NaN
codes["technology"] = codes["CPC_Class"].str[9].where(is_renewable).map(TECHNOLOGIES)

# Add the priority year to each code
codes = codes.merge(patents, on="appln_id", how="left")

In [8]:
# One row per patent and category (main group or technology), so that a patent counts once
# in each category it belongs to, no matter how many of its codes fall in that category
categories = pd.concat([
    codes[["appln_id", "prio_year", "main_group"]].rename(columns={"main_group": "category"}),
    codes[["appln_id", "prio_year", "technology"]].rename(columns={"technology": "category"}),
]).dropna().drop_duplicates().reset_index(drop=True)
categories["category"] = categories["category"].astype("category")

# Applicant names that differ only in capitalization are the same applicant
applicants["app_name"] = applicants["app_name"].str.upper()

In [9]:
def check(label, actual, expected):
    """Print OK or MISMATCH for a checkpoint from the task text."""
    status = "OK" if actual == expected else "MISMATCH"
    print(f"{status:9}{label}: got {actual:,}, expected {expected:,}")

check("Patents in the sample", len(patents), 2_835_599)
check("Green energy patents", codes["appln_id"].nunique(), 114_924)
check("Sum of app_share", round(applicants["app_share"].sum()), 2_835_140)

OK       Patents in the sample: got 2,835,599, expected 2,835,599
OK       Green energy patents: got 114,924, expected 114,924
OK       Sum of app_share: got 2,835,140, expected 2,835,140


In [10]:
# Overview numbers
n_green = codes["appln_id"].nunique()
avg_codes = len(codes) / n_green
multi_group = (codes.groupby("appln_id")["main_group"].nunique() > 1).sum()
multi_country = (applicants.groupby("appln_id")["ctry_code"].nunique() > 1).sum()

print(f"Average number of Y02E codes per green energy patent: {avg_codes:.2f}")
print(f"Green energy patents in more than one main group: {multi_group:,}")
print(f"Patents with applicants in more than one country: {multi_country:,}")

Average number of Y02E codes per green energy patent: 1.23
Green energy patents in more than one main group: 4,359
Patents with applicants in more than one country: 58,955


**Overview of the data.**

* All three checkpoints are reproduced: 2,835,599 patents in the sample, 114,924 green energy patents, and a sum of `app_share` of 2,835,140.
* A green energy patent has on average **1.23** Y02E codes, so most green patents carry a single Y02E code.
* **4,359** green energy patents belong to more than one main group (about 3.8% of the green patents), so the main-group shares in Task 3 count some patents twice.
* **58,955** patents in the sample have applicants in more than one country, which is why fractional counting by `app_share` is used in Task 5.

In [11]:
# Tasks 2–5 only need applicants of green energy patents; keep only those to save memory
applicants = applicants[applicants["appln_id"].isin(codes["appln_id"])].reset_index(drop=True)
applicants["ctry_code"] = applicants["ctry_code"].astype("category")

# Memory use of the prepared data, in MB
for name, df in {"patents": patents, "codes": codes, "categories": categories, "applicants": applicants}.items():
    print(f"{name:11}{len(df):>10,} rows {df.memory_usage(deep=True).sum() / 1e6:8.1f} MB")

patents     2,835,599 rows     17.0 MB
codes         141,882 rows     24.7 MB
categories    162,166 rows      1.1 MB
applicants    126,097 rows     11.2 MB


---
## ⚠️ Explanation of Task 1 — delete this section before submission
The cell below walks through every function and object created in Task 1, using a few raw lines from the files and one example patent.

In [12]:
# ==========================================================================================
# EXPLANATION CELL — FOR YOUR OWN UNDERSTANDING. DELETE THIS CELL BEFORE SUBMISSION.
# It re-reads only a few raw lines from each file, and otherwise uses the objects created above.
# ==========================================================================================
from IPython.display import display

def heading(text):
    """Print a section heading for the walkthrough."""
    print("\n" + "=" * 90 + f"\n{text}\n" + "=" * 90)

# ---- 0. Environment ------------------------------------------------------------------------
heading("0. ENVIRONMENT")
import sys
print("Python running this notebook:", sys.executable)
print("pandas version:", pd.__version__, "| numpy version:", np.__version__)
print("If this path does not contain 'miniforge', VS Code has picked the wrong kernel.")

# ---- 1. Constants ----------------------------------------------------------------------------
heading("1. CONSTANTS (dictionaries = lookup tables)")
print("MAIN_GROUPS turns the two digits before the slash into a label (Table 1):")
print("   '10' ->", MAIN_GROUPS["10"], "|  '60' ->", MAIN_GROUPS["60"])
print("TECHNOLOGIES turns the first digit after the slash (main group 10 only) into a technology (Table 2):")
print("   '5' ->", TECHNOLOGIES["5"], "|  '7' ->", TECHNOLOGIES["7"], "|  '0' ->", TECHNOLOGIES.get("0"), "(not in Table 2)")
print(f"CHUNK_SIZE = {CHUNK_SIZE:,}: the CPC file (35.6 million rows) is read in about "
      f"{-(-35_600_000 // CHUNK_SIZE)} pieces, so only one piece of 2 million rows is in memory at a time.")

# ---- 2. read_sample ------------------------------------------------------------------------
heading("2. read_sample()  ->  patents")
raw_ipc = pd.read_csv(DATA_DIR + "202608_EPO_IPC.txt", sep="|", nrows=6)
print("RAW file, first 6 rows. Note: appln_id 1 is repeated once per IPC code,")
print("and its priority year 1999 is outside the sample:")
display(raw_ipc)
print("RESULT `patents`: one row per patent, only the two columns we need, years 2000-2019:")
display(patents.head())
print(f"Rows in raw file: about 19.7 million  ->  rows in `patents`: {len(patents):,}")
print("dtypes:", patents.dtypes.to_dict(), "(int32/int16 use half/quarter the memory of int64)")

# ---- 3. read_green_codes + parsing ------------------------------------------------------------
heading("3. read_green_codes()  ->  codes")
raw_cpc = pd.read_csv(DATA_DIR + "202608_CPC_Classes.txt", sep="|", nrows=5)
print("RAW file, first 5 rows. Almost all codes are NOT green (Y02E), so they are thrown away chunk by chunk:")
display(raw_cpc)
example_code = codes["CPC_Class"].iloc[0]
print(f"How a code is parsed, using {example_code!r} (note the TWO spaces after Y02E):")
for i, ch in enumerate(example_code):
    print(f"   position {i:2}: {ch!r}")
print("   .str[6:8] -> characters 6 and 7 = main group | .str[9] -> character 9 = technology digit")
print("\nRESULT `codes`: one row per Y02E code, with main group, technology and priority year added:")
display(codes.head())
print(f"{len(codes):,} Y02E codes on {codes['appln_id'].nunique():,} green energy patents.")
print("technology is NaN (missing) for codes outside main group 10, and for 'Y02E  10/00', which is not in Table 2.")

# ---- 4. Pick one example patent and follow it through every table -----------------------------
heading("4. ONE EXAMPLE PATENT THROUGH ALL TABLES")
codes_per_tech = codes.dropna(subset=["technology"]).groupby(["appln_id", "technology"]).size()
several_codes = codes_per_tech[codes_per_tech > 1].index.get_level_values("appln_id")
countries_per_patent = applicants.groupby("appln_id")["ctry_code"].nunique()
several_countries = countries_per_patent[countries_per_patent > 1].index
example_id = several_codes.intersection(several_countries)[0]
print(f"Example: appln_id {example_id}, chosen because it has several codes in one technology")
print("AND applicants in more than one country.\n")
print("In `patents` (one row):")
display(patents[patents["appln_id"] == example_id])
print("In `codes` (one row per Y02E code):")
display(codes[codes["appln_id"] == example_id])
print("In `categories` (one row per category, duplicates removed -> counts ONCE per category):")
display(categories[categories["appln_id"] == example_id])
print("In `applicants` (one row per applicant; app_share adds up to 1 per patent):")
example_apps = applicants[applicants["appln_id"] == example_id]
display(example_apps)
print("Sum of app_share for this patent:", round(example_apps["app_share"].sum(), 6))

# ---- 5. read_applicants and the duplicate rows -----------------------------------------------
heading("5. read_applicants()  ->  why drop_duplicates() is needed")
raw_app = pd.read_csv(DATA_DIR + "202608_EPO_App_reg.txt", sep="|", encoding="utf-8-sig", nrows=200_000)
kept_cols = ["appln_id", "app_name", "ctry_code", "app_share"]
dup_id = raw_app.loc[raw_app.duplicated(subset=kept_cols, keep=False), "appln_id"].iloc[0]
print(f"RAW rows for appln_id {dup_id}: the same applicant appears twice because the address is linked")
print("to two regions (see reg_code). If both rows were kept, this applicant's share would count twice.")
display(raw_app.loc[raw_app["appln_id"] == dup_id, kept_cols + ["reg_code", "reg_share"]])
print("After reading only the four columns we need, these rows are identical, so drop_duplicates() removes one.")
print("Names are converted to upper case so that e.g. 'Siemens AG' and 'SIEMENS AG' become one applicant.")

# ---- 6. Checkpoints and overview numbers -----------------------------------------------------
heading("6. WHAT THE OVERVIEW NUMBERS MEAN")
print("len(codes) / number of green patents = average number of Y02E codes per green patent.")
print("groupby('appln_id')[col].nunique() counts DIFFERENT values per patent, e.g. main groups per patent:")
display(codes.groupby("appln_id")["main_group"].nunique().value_counts().rename("number of patents"))
print("'> 1' turns that into True/False, and .sum() counts the Trues (True = 1, False = 0).")
print("Countries per patent (sample-wide count, computed before applicants were reduced to green patents):")
print(f"   patents with applicants in more than one country = {multi_country:,}")

# ---- 7. What Tasks 2-5 will use ------------------------------------------------------------
heading("7. WHAT EACH OBJECT IS FOR IN TASKS 2-5")
print("patents     -> Task 2: number of ALL patents per year")
print("codes       -> Task 1 overview numbers (one row per code)")
print("categories  -> Tasks 2-5: count green patents per year / period / main group / technology")
print("              (Task 2 green count: categories['appln_id'].nunique() per year, since a patent")
print("               can have several categories)")
print("applicants  -> Task 5: fractional counts (sum of app_share) per country or applicant")



0. ENVIRONMENT
Python running this notebook: /sessions/rcw-01v5btf7pcipgw7dzq989lxv/work/env313/bin/python
pandas version: 3.1.1 | numpy version: 2.5.5
If this path does not contain 'miniforge', VS Code has picked the wrong kernel.

1. CONSTANTS (dictionaries = lookup tables)
MAIN_GROUPS turns the two digits before the slash into a label (Table 1):
   '10' -> Renewable energy |  '60' -> Storage and hydrogen
TECHNOLOGIES turns the first digit after the slash (main group 10 only) into a technology (Table 2):
   '5' -> Solar |  '7' -> Wind |  '0' -> None (not in Table 2)
CHUNK_SIZE = 2,000,000: the CPC file (35.6 million rows) is read in about 18 pieces, so only one piece of 2 million rows is in memory at a time.

2. read_sample()  ->  patents
RAW file, first 6 rows. Note: appln_id 1 is repeated once per IPC code,
and its priority year 1999 is outside the sample:


,appln_id,prio_year,app_year,IPC
0,1,1999,2000,G06K007/00
1,1,1999,2000,G06K017/00
2,1,1999,2000,G06K019/077
3,1,1999,2000,H01R012/18
4,1,1999,2000,H04M001/02
5,1,1999,2000,H04M001/2745


RESULT `patents`: one row per patent, only the two columns we need, years 2000-2019:


,appln_id,prio_year
0,8,2000
1,10,2000
2,11,2000
3,12,2000
4,13,2000


Rows in raw file: about 19.7 million  ->  rows in `patents`: 2,835,599
dtypes: {'appln_id': dtype('int32'), 'prio_year': dtype('int16')} (int32/int16 use half/quarter the memory of int64)

3. read_green_codes()  ->  codes
RAW file, first 5 rows. Almost all codes are NOT green (Y02E), so they are thrown away chunk by chunk:


,appln_id,CPC_Class
0,1,G06K 7/0013
1,1,G06K 7/0021
2,1,G06K 7/0043
3,1,G06K 7/0069
4,1,G06K 19/07739


... (12 outputs left out)

,appln_id,app_name,ctry_code,app_share,reg_code,reg_share
1044,1252,Arçelik A.S.,TR,1.0,TR423,0.895
1045,1252,Arçelik A.S.,TR,1.0,TR905,0.105


After reading only the four columns we need, these rows are identical, so drop_duplicates() removes one.
Names are converted to upper case so that e.g. 'Siemens AG' and 'SIEMENS AG' become one applicant.

6. WHAT THE OVERVIEW NUMBERS MEAN
len(codes) / number of green patents = average number of Y02E codes per green patent.
groupby('appln_id')[col].nunique() counts DIFFERENT values per patent, e.g. main groups per patent:


main_group
1    110565
2      3457
3       832
4        70
Name: number of patents, dtype: int64

'> 1' turns that into True/False, and .sum() counts the Trues (True = 1, False = 0).
Countries per patent (sample-wide count, computed before applicants were reduced to green patents):
   patents with applicants in more than one country = 58,955

7. WHAT EACH OBJECT IS FOR IN TASKS 2-5
patents     -> Task 2: number of ALL patents per year
codes       -> Task 1 overview numbers (one row per code)
categories  -> Tasks 2-5: count green patents per year / period / main group / technology
              (Task 2 green count: categories['appln_id'].nunique() per year, since a patent
               can have several categories)
applicants  -> Task 5: fractional counts (sum of app_share) per country or applicant
